# Notebook 05 — Database Design, Storage and Secure Access

**ST5011CEM Big Data Programming Project**

Designs and populates a normalised relational store for the analytical results,
and demonstrates secure query practice.

**Why a database as well as Parquet.** Parquet suits full-column analytical scans
across the pipeline. A relational store suits the stakeholder-facing workload:
indexed point lookups ("how is operator X performing on route Y?"), referential
integrity between entities, and access from tools that speak SQL rather than
Spark. The two are complementary, and the brief asks for both.

Covers these brief requirements:

| Requirement | Where |
|---|---|
| Database design and implementation | §2 |
| Relationships between datasets | §2, §4 |
| Parameterised queries (no string concatenation) | §5 |
| SQL injection prevention | §5 |
| Integration of multiple datasets using joins | §6 |
| Data persistence strategies | §3, §8 |
| SQL dump and schema diagram for submission | §8 |

## 1. Setup

In [1]:
import os, sys, sqlite3, time
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pathlib import Path
import pandas as pd

from pyspark.sql import SparkSession, functions as F

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/raw")

PROJECT = find_project_root(Path.cwd())
DB_DIR  = PROJECT / "data" / "db"
DOCS    = PROJECT / "docs"
DB_DIR.mkdir(parents=True, exist_ok=True)
DOCS.mkdir(parents=True, exist_ok=True)

DB_PATH = DB_DIR / "bus_analytics.db"
print("Database:", DB_PATH)

spark = (SparkSession.builder
         .appName("ST5011CEM_Database")
         .master("local[*]")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.adaptive.enabled", "false")
         .config("spark.sql.session.timeZone", "Europe/London")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

delays = spark.read.parquet((PROJECT / "data" / "processed" / "observed_delays").as_posix())
print(f"Source observations: {delays.count():,}")

Database: E:\BODS-project\data\db\bus_analytics.db
Source observations: 1,201,509


## 2. Schema design

Third normal form, with the fact table referencing three dimension tables.
Storing operator and route names once rather than repeating them on every
observation removes update anomalies and shrinks the fact table considerably.

```
  operators ──┐
              ├──< delay_observations >── stops
  routes    ──┘
```

`PRAGMA foreign_keys = ON` is required — SQLite does not enforce foreign keys by
default, which is a common and easily missed source of orphaned rows.

In [2]:
SCHEMA = """
PRAGMA foreign_keys = ON;

DROP TABLE IF EXISTS delay_observations;
DROP TABLE IF EXISTS routes;
DROP TABLE IF EXISTS stops;
DROP TABLE IF EXISTS operators;
DROP TABLE IF EXISTS model_results;

CREATE TABLE operators (
    operator_id    INTEGER PRIMARY KEY,
    agency_id      TEXT    NOT NULL UNIQUE,
    operator_name  TEXT    NOT NULL
);

CREATE TABLE routes (
    route_id       INTEGER PRIMARY KEY,
    route_code     TEXT    NOT NULL,
    operator_id    INTEGER NOT NULL,
    UNIQUE (route_code, operator_id),
    FOREIGN KEY (operator_id) REFERENCES operators(operator_id)
);

CREATE TABLE stops (
    stop_pk        INTEGER PRIMARY KEY,
    stop_id        TEXT    NOT NULL UNIQUE,
    latitude       REAL    NOT NULL,
    longitude      REAL    NOT NULL
);

CREATE TABLE delay_observations (
    observation_id INTEGER PRIMARY KEY AUTOINCREMENT,
    vehicle_ref    TEXT    NOT NULL,
    route_id       INTEGER NOT NULL,
    stop_pk        INTEGER NOT NULL,
    observed_at    TEXT    NOT NULL,
    obs_date       TEXT    NOT NULL,
    scheduled_sec  INTEGER NOT NULL,
    observed_sec   INTEGER NOT NULL,
    delay_min      REAL    NOT NULL,
    stop_sequence  INTEGER,
    match_dist_m   REAL,
    on_time_2min   INTEGER NOT NULL CHECK (on_time_2min IN (0, 1)),
    FOREIGN KEY (route_id) REFERENCES routes(route_id),
    FOREIGN KEY (stop_pk)  REFERENCES stops(stop_pk)
);

CREATE TABLE model_results (
    result_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    model_name     TEXT    NOT NULL,
    rmse           REAL,
    mae            REAL,
    r2             REAL,
    train_seconds  REAL,
    recorded_at    TEXT    DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX idx_obs_route  ON delay_observations(route_id);
CREATE INDEX idx_obs_stop   ON delay_observations(stop_pk);
CREATE INDEX idx_obs_date   ON delay_observations(obs_date);
CREATE INDEX idx_obs_ontime ON delay_observations(on_time_2min);
CREATE INDEX idx_routes_op  ON routes(operator_id);
"""

if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(DB_PATH)
conn.executescript(SCHEMA)
conn.commit()

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
print("Tables created:")
print(tables.to_string(index=False))

Tables created:
              name
delay_observations
     model_results
         operators
            routes
   sqlite_sequence
             stops


## 3. Populating the dimension tables

Dimensions are small, so they are collected to the driver and inserted with
`executemany`. The fact table is handled in chunks in §4.

In [3]:
ops = (delays.select("agency_id", "agency_name").distinct()
       .filter(F.col("agency_id").isNotNull()).toPandas())
ops["operator_id"] = range(1, len(ops) + 1)

conn.executemany(
    "INSERT INTO operators (operator_id, agency_id, operator_name) VALUES (?, ?, ?)",
    ops[["operator_id", "agency_id", "agency_name"]].itertuples(index=False, name=None))
conn.commit()
print(f"operators: {len(ops):,}")

rts = (delays.select("route_short_name", "agency_id").distinct()
       .filter(F.col("route_short_name").isNotNull()).toPandas())
rts = rts.merge(ops[["agency_id", "operator_id"]], on="agency_id", how="inner")
rts["route_id"] = range(1, len(rts) + 1)

conn.executemany(
    "INSERT INTO routes (route_id, route_code, operator_id) VALUES (?, ?, ?)",
    rts[["route_id", "route_short_name", "operator_id"]].itertuples(index=False, name=None))
conn.commit()
print(f"routes: {len(rts):,}")

stp = (delays.select("stop_id", "stop_lat", "stop_lon").distinct()
       .filter(F.col("stop_id").isNotNull()).toPandas())
stp["stop_pk"] = range(1, len(stp) + 1)

conn.executemany(
    "INSERT INTO stops (stop_pk, stop_id, latitude, longitude) VALUES (?, ?, ?, ?)",
    stp[["stop_pk", "stop_id", "stop_lat", "stop_lon"]].itertuples(index=False, name=None))
conn.commit()
print(f"stops: {len(stp):,}")

operators: 29
routes: 425
stops: 14,741


## 4. Loading the fact table

The observation set is too large to collect to the driver in one piece, so it is
written in chunks. Surrogate keys are resolved by joining in Spark before the
data leaves the cluster — doing it row by row in Python would be far slower.

In [4]:
ops_sdf = spark.createDataFrame(ops[["agency_id", "operator_id"]])
rts_sdf = spark.createDataFrame(rts[["route_short_name", "operator_id", "route_id"]])
stp_sdf = spark.createDataFrame(stp[["stop_id", "stop_pk"]])

facts = (delays
         .drop("route_id")  # GTFS route_id replaced by DB surrogate key
         .join(F.broadcast(ops_sdf), "agency_id")
         .join(F.broadcast(rts_sdf), ["route_short_name", "operator_id"])
         .join(F.broadcast(stp_sdf), "stop_id")
         .select(
             F.col("vehicle_ref"),
             F.col("route_id"),
             F.col("stop_pk"),
             F.date_format("recorded_ts", "yyyy-MM-dd HH:mm:ss").alias("observed_at"),
             F.date_format("recorded_ts", "yyyy-MM-dd").alias("obs_date"),
             F.col("arrival_sec").cast("int").alias("scheduled_sec"),
             F.col("obs_sec").cast("int").alias("observed_sec"),
             F.round("delay_min", 4).alias("delay_min"),
             F.col("stop_sequence").cast("int"),
             F.round("dist_m", 2).alias("match_dist_m"),
             (F.abs(F.col("delay_min")) <= 2).cast("int").alias("on_time_2min"),
         ))

total = facts.count()
print(f"Rows to insert: {total:,}")

INSERT = """INSERT INTO delay_observations
    (vehicle_ref, route_id, stop_pk, observed_at, obs_date,
     scheduled_sec, observed_sec, delay_min, stop_sequence,
     match_dist_m, on_time_2min)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)"""

t0 = time.time()
CHUNK = 50_000
written = 0
for pdf in facts.toPandas().groupby(facts.toPandas().index // CHUNK):
    rows = pdf[1].where(pd.notnull(pdf[1]), None).itertuples(index=False, name=None)
    conn.executemany(INSERT, rows)
    written += len(pdf[1])
    print(f"  {written:,} / {total:,}")
conn.commit()
print(f"\nLoaded in {time.time()-t0:.1f}s")

Rows to insert: 1,201,509
  50,000 / 1,201,509
  100,000 / 1,201,509
  150,000 / 1,201,509
  200,000 / 1,201,509
  250,000 / 1,201,509
  300,000 / 1,201,509
  350,000 / 1,201,509
  400,000 / 1,201,509
  450,000 / 1,201,509
  500,000 / 1,201,509
  550,000 / 1,201,509
  600,000 / 1,201,509
  650,000 / 1,201,509
  700,000 / 1,201,509
  750,000 / 1,201,509
  800,000 / 1,201,509
  850,000 / 1,201,509
  900,000 / 1,201,509
  950,000 / 1,201,509
  1,000,000 / 1,201,509
  1,050,000 / 1,201,509
  1,100,000 / 1,201,509
  1,150,000 / 1,201,509
  1,200,000 / 1,201,509
  1,201,509 / 1,201,509

Loaded in 64.9s


In [5]:
# Store the model comparison results alongside the data
cmp_path = DOCS / "model_comparison.csv"
if cmp_path.exists():
    cmp = pd.read_csv(cmp_path)
    conn.executemany(
        """INSERT INTO model_results (model_name, rmse, mae, r2, train_seconds)
           VALUES (?, ?, ?, ?, ?)""",
        cmp[["model", "rmse", "mae", "r2", "train_secs"]].itertuples(index=False, name=None))
    conn.commit()
    print("Model results stored.")
else:
    print("model_comparison.csv not found - run notebook 04 first.")

print("\nRow counts:")
for t in ["operators", "routes", "stops", "delay_observations", "model_results"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:<22}{n:>10,}")

Model results stored.

Row counts:
  operators                     29
  routes                       425
  stops                     14,741
  delay_observations     1,201,509
  model_results                  3


## 5. Secure query practice

The brief requires parameterised queries and no string concatenation. The
difference matters: string interpolation lets input become executable SQL,
whereas parameter binding passes it to the driver as a value that can never be
parsed as code.

The demonstration below is run against this project's own database to show the
mechanism concretely.

In [6]:
def get_operator_stats_UNSAFE(operator_name: str):
    """VULNERABLE - shown to illustrate the risk. Never used in this project."""
    query = ("SELECT operator_name, COUNT(*) AS n "
             "FROM delay_observations o "
             "JOIN routes r ON o.route_id = r.route_id "
             "JOIN operators op ON r.operator_id = op.operator_id "
             f"WHERE op.operator_name = '{operator_name}' "
             "GROUP BY operator_name")
    return query


def get_operator_stats_SAFE(conn, operator_name: str):
    """Parameterised. The ? placeholder is bound as a value, never parsed as SQL."""
    query = """
        SELECT op.operator_name,
               COUNT(*)                                       AS observations,
               ROUND(AVG(o.delay_min), 3)                     AS mean_delay_min,
               ROUND(100.0 * SUM(o.on_time_2min) / COUNT(*), 2) AS reliability_pct
        FROM delay_observations o
        JOIN routes    r  ON o.route_id    = r.route_id
        JOIN operators op ON r.operator_id = op.operator_id
        WHERE op.operator_name = ?
        GROUP BY op.operator_name
    """
    return pd.read_sql_query(query, conn, params=(operator_name,))


malicious = "x' OR '1'='1"

print("Unsafe construction produces this SQL:")
print(" ", get_operator_stats_UNSAFE(malicious).split("WHERE")[1].strip())
print("  -> the WHERE clause is always true; the filter is bypassed entirely.\n")

print("Parameterised version with the same input:")
result = get_operator_stats_SAFE(conn, malicious)
print(f"  rows returned: {len(result)}  (the string is treated as a literal name)")

Unsafe construction produces this SQL:
  op.operator_name = 'x' OR '1'='1' GROUP BY operator_name
  -> the WHERE clause is always true; the filter is bypassed entirely.

Parameterised version with the same input:
  rows returned: 0  (the string is treated as a literal name)


In [7]:
# The same function with legitimate input
sample_op = conn.execute("SELECT operator_name FROM operators LIMIT 1").fetchone()[0]
print(f"Legitimate query for '{sample_op}':")
get_operator_stats_SAFE(conn, sample_op)

Legitimate query for 'D & G Bus':


,operator_name,observations,mean_delay_min,reliability_pct
0,D & G Bus,3042,0.765,36.09


In [8]:
def reliability_by_route(conn, operator_name: str, min_observations: int = 50):
    """Multiple bound parameters; still no concatenation."""
    query = """
        SELECT r.route_code                                    AS route,
               COUNT(*)                                        AS observations,
               ROUND(AVG(o.delay_min), 3)                      AS mean_delay_min,
               ROUND(100.0*SUM(o.on_time_2min)/COUNT(*), 2)    AS reliability_pct
        FROM delay_observations o
        JOIN routes    r  ON o.route_id    = r.route_id
        JOIN operators op ON r.operator_id = op.operator_id
        WHERE op.operator_name = ?
        GROUP BY r.route_code
        HAVING COUNT(*) >= ?
        ORDER BY reliability_pct ASC
        LIMIT 15
    """
    return pd.read_sql_query(query, conn, params=(operator_name, min_observations))

busiest = conn.execute("""
    SELECT op.operator_name FROM delay_observations o
    JOIN routes r ON o.route_id = r.route_id
    JOIN operators op ON r.operator_id = op.operator_id
    GROUP BY op.operator_name ORDER BY COUNT(*) DESC LIMIT 1""").fetchone()[0]

print(f"Worst-performing routes for {busiest}:")
reliability_by_route(conn, busiest)

Worst-performing routes for Bee Network:


,route,observations,mean_delay_min,reliability_pct
0,221,168,1.543,10.71
1,487,644,2.267,12.27
2,630,779,2.563,16.05
3,341,2329,3.858,16.19
4,356,4217,2.054,16.34
5,634,244,2.806,18.85
6,415,1929,2.932,19.34
7,382,1751,1.168,21.87
8,475,1364,2.749,22.29
9,477,1804,3.830,23.45


## 6. Stakeholder queries

Four-table joins answering the questions a transport authority would actually
ask. These are the "sample queries" the submission requires.

In [9]:
Q_OPERATOR_LEAGUE = """
    SELECT op.operator_name                                  AS operator,
           COUNT(DISTINCT r.route_id)                        AS routes_operated,
           COUNT(*)                                          AS observations,
           ROUND(AVG(o.delay_min), 3)                        AS mean_delay_min,
           ROUND(100.0*SUM(o.on_time_2min)/COUNT(*), 2)      AS reliability_pct,
           CASE WHEN 100.0*SUM(o.on_time_2min)/COUNT(*) >= 85
                THEN 'COMPLIANT' ELSE 'BELOW THRESHOLD' END  AS status
    FROM delay_observations o
    JOIN routes    r  ON o.route_id    = r.route_id
    JOIN operators op ON r.operator_id = op.operator_id
    GROUP BY op.operator_name
    HAVING COUNT(*) >= ?
    ORDER BY reliability_pct DESC
"""
league = pd.read_sql_query(Q_OPERATOR_LEAGUE, conn, params=(100,))
print("Operator compliance league table:")
league

Operator compliance league table:


,operator,routes_operated,observations,mean_delay_min,reliability_pct,status
0,Howards Travel,2,225,-0.356,67.11,BELOW THRESHOLD
1,Stagecoach Cumbria and North Lancashire,5,16537,0.119,60.17,BELOW THRESHOLD
2,Bee Network,264,1023236,0.460,55.59,BELOW THRESHOLD
3,The Blackburn Bus Company,5,21867,0.706,54.34,BELOW THRESHOLD
4,Arriva North West,36,61592,0.471,47.26,BELOW THRESHOLD
5,Hattons Travel,7,680,2.411,46.62,BELOW THRESHOLD
6,Warrington's Own Buses,38,34997,1.038,45.36,BELOW THRESHOLD
7,First Halifax,1,2093,1.045,43.10,BELOW THRESHOLD
8,Ashcroft Travel,6,1950,2.192,38.41,BELOW THRESHOLD
9,Stagecoach Merseyside and South Lancashire,5,1535,0.791,37.85,BELOW THRESHOLD


In [10]:
Q_WORST_STOPS = """
    SELECT s.stop_id,
           ROUND(s.latitude, 5)                  AS lat,
           ROUND(s.longitude, 5)                 AS lon,
           COUNT(*)                              AS observations,
           ROUND(AVG(o.delay_min), 3)            AS mean_delay_min
    FROM delay_observations o
    JOIN stops s ON o.stop_pk = s.stop_pk
    GROUP BY s.stop_id, s.latitude, s.longitude
    HAVING COUNT(*) >= ?
    ORDER BY mean_delay_min DESC
    LIMIT ?
"""
print("Stops with the highest mean delay (intervention candidates):")
pd.read_sql_query(Q_WORST_STOPS, conn, params=(30, 15))

Stops with the highest mean delay (intervention candidates):


,stop_id,lat,lon,observations,mean_delay_min
0,1800ED30641,53.54802,-2.06893,32,13.186
1,1800ED22661,53.54324,-2.13440,30,12.329
2,450019258,53.59952,-1.92885,33,11.613
3,1800ED19041,53.54589,-2.08168,30,11.173
4,068000000575,53.33342,-2.70479,33,11.098
5,1800NE07871,53.54003,-2.18649,31,10.816
6,1800NE07951,53.53913,-2.18307,31,10.758
7,1800EB35061,53.49715,-2.18385,30,10.667
8,1800ED22581,53.53784,-2.14294,31,10.078
9,1800ED08561,53.52784,-2.12625,41,9.833


In [11]:
Q_HOURLY = """
    SELECT CAST(strftime('%H', observed_at) AS INTEGER)      AS hour,
           COUNT(*)                                          AS observations,
           ROUND(AVG(delay_min), 3)                          AS mean_delay_min,
           ROUND(100.0*SUM(on_time_2min)/COUNT(*), 2)        AS reliability_pct
    FROM delay_observations
    GROUP BY hour
    ORDER BY hour
"""
print("Network performance by hour:")
pd.read_sql_query(Q_HOURLY, conn)

Network performance by hour:


,hour,observations,mean_delay_min,reliability_pct
0,0,5,-9.563,20.00
1,1,2,1.208,0.00
2,3,1,8.950,0.00
3,4,228,-5.069,51.75
4,5,4374,-0.724,53.02
5,6,33,0.184,45.45
6,7,31259,-0.175,55.93
7,8,16512,-0.099,56.46
8,9,310,-0.247,56.77
9,10,76502,0.486,54.67


## 7. Index effectiveness

`EXPLAIN QUERY PLAN` confirms the optimiser uses the indexes created in §2 rather
than scanning the fact table.

In [12]:
plan = pd.read_sql_query(
    """EXPLAIN QUERY PLAN
       SELECT COUNT(*) FROM delay_observations
       WHERE obs_date = ? AND on_time_2min = ?""",
    conn, params=("2026-07-24", 1))
print("Query plan:")
print(plan.to_string(index=False))

t0 = time.time()
conn.execute("SELECT COUNT(*) FROM delay_observations WHERE on_time_2min = 1").fetchone()
indexed = time.time() - t0

t0 = time.time()
conn.execute("SELECT COUNT(*) FROM delay_observations WHERE match_dist_m < 20").fetchone()
unindexed = time.time() - t0

print(f"\nIndexed column filter   : {indexed*1000:7.2f} ms")
print(f"Unindexed column filter : {unindexed*1000:7.2f} ms")

Query plan:
 id  parent  notused                                                                detail
  4       0        0 SEARCH delay_observations USING INDEX idx_obs_ontime (on_time_2min=?)

Indexed column filter   :   28.41 ms
Unindexed column filter :  184.49 ms


## 8. Export for submission

The brief requires a SQL dump, a schema diagram and sample queries. The dump and
the queries are written here; the diagram is generated in §9.

In [13]:
dump_path = DB_DIR / "bus_analytics_dump.sql"
with open(dump_path, "w", encoding="utf-8") as fh:
    for line in conn.iterdump():
        fh.write(f"{line}\n")
print(f"SQL dump: {dump_path}  ({dump_path.stat().st_size/1e6:.1f} MB)")

queries_path = DOCS / "sample_queries.sql"
with open(queries_path, "w", encoding="utf-8") as fh:
    fh.write("-- ST5011CEM sample queries\n")
    fh.write("-- All application queries use bound parameters (?) rather than\n")
    fh.write("-- string concatenation, preventing SQL injection.\n\n")
    for name, q in [("Operator compliance league table", Q_OPERATOR_LEAGUE),
                    ("Stops with highest mean delay",    Q_WORST_STOPS),
                    ("Network performance by hour",      Q_HOURLY)]:
        fh.write(f"-- {name}\n{q.strip()}\n\n")
print(f"Sample queries: {queries_path}")

league.to_csv(DOCS / "operator_league_table.csv", index=False)
print("League table exported.")

SQL dump: E:\BODS-project\data\db\bus_analytics_dump.sql  (159.9 MB)
Sample queries: E:\BODS-project\docs\sample_queries.sql
League table exported.


## 9. Schema diagram

In [14]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

FIGURES = DOCS / "figures"; FIGURES.mkdir(parents=True, exist_ok=True)

entities = {
    "operators": (0.06, 0.62, ["operator_id  PK", "agency_id  UQ", "operator_name"]),
    "routes":    (0.06, 0.18, ["route_id  PK", "route_code", "operator_id  FK"]),
    "stops":     (0.72, 0.62, ["stop_pk  PK", "stop_id  UQ", "latitude", "longitude"]),
    "delay_observations": (0.38, 0.30,
        ["observation_id  PK", "vehicle_ref", "route_id  FK", "stop_pk  FK",
         "observed_at", "delay_min", "on_time_2min"]),
}

fig, ax = plt.subplots(figsize=(11, 6.5))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")

boxes = {}
for name, (x, y, cols) in entities.items():
    h = 0.055 + 0.038 * len(cols)
    w = 0.245
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.006",
                                fc="#eef3fb", ec="#2b5d9e", lw=1.6))
    ax.text(x + w/2, y + h - 0.032, name, ha="center", va="center",
            fontsize=10.5, fontweight="bold", color="#14213d")
    for i, col in enumerate(cols):
        ax.text(x + 0.014, y + h - 0.068 - i*0.038, col,
                ha="left", va="center", fontsize=8.2, family="monospace")
    boxes[name] = (x, y, w, h)

def link(a, b):
    ax_, ay, aw, ah = boxes[a]; bx, by, bw, bh = boxes[b]
    ax.add_patch(FancyArrowPatch((ax_ + aw/2, ay + ah/2), (bx + bw/2, by + bh/2),
                                 arrowstyle="-|>", mutation_scale=15,
                                 color="#2b5d9e", lw=1.3,
                                 connectionstyle="arc3,rad=0.12"))

link("routes", "operators")
link("delay_observations", "routes")
link("delay_observations", "stops")

ax.set_title("Database Schema — Bus Delay Analytics", fontsize=13, pad=14)
fig.savefig(FIGURES / "fig14_schema_diagram.png", dpi=140, bbox_inches="tight")
plt.close(fig)
print("saved fig14_schema_diagram.png")

saved fig14_schema_diagram.png


In [15]:
conn.close()
spark.stop()
print("Notebook 05 complete.")
print(f"\nDatabase : {DB_PATH}")
print(f"SQL dump : {DB_DIR / 'bus_analytics_dump.sql'}")
print("Diagram  : docs/figures/fig14_schema_diagram.png")

Notebook 05 complete.

Database : E:\BODS-project\data\db\bus_analytics.db
SQL dump : E:\BODS-project\data\db\bus_analytics_dump.sql
Diagram  : docs/figures/fig14_schema_diagram.png
